# 3. Deployment, Inference, Evaluation, and Monitoring

Deploy a registered candidate to an Azure ML managed online endpoint, evaluate it on sealed test data, promote it through an explicit quality gate, explain a scalar quality target with SHAP, and establish drift monitoring.

## Production workflow

A candidate deployment starts with **zero traffic**. Direct invocation validates readiness and held-out behavior before promotion. The gate combines answer token-F1 and mean model latency; add groundedness, safety, fairness, load, and cost thresholds appropriate to the use case. Keep the prior deployment available for rollback.

> SHAP does not intrinsically explain free-form text generation. This notebook explains token contributions to a declared scalar target: reference-answer token-F1. Treat it as diagnostic evidence, not a causal explanation.

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path
import sys

from dotenv import load_dotenv
import mlflow
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lib").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "lib").exists():
    raise RuntimeError("Start this notebook from the repository or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from lib.azureml_ops import (
    create_inference_environment,
    deploy_managed_endpoint,
    promote_deployment,
)
from lib.config import AzureMLConfig
from lib.data import read_jsonl
from lib.evaluation import token_f1
from lib.explainability import explain_text_score
from lib.inference import evaluate_endpoint, invoke_endpoint
from lib.monitoring import create_reference_profile, detect_drift, write_json_artifact

## Deployment and gate configuration

Endpoint names must be globally unique in the Azure region. Use a GPU instance sized from load tests, not notebook convenience. The initial values below are candidate defaults, not universal service-level objectives.

In [ ]:
REGISTERED_MODEL_NAME = "raft-llama32-1b"
REGISTERED_MODEL_VERSION = "1"
ENDPOINT_NAME = "sriks-raft-llama32-1b"
DEPLOYMENT_NAME = f"candidate-{REGISTERED_MODEL_VERSION.replace('.', '-')}"
INFERENCE_ENVIRONMENT_NAME = "raft-online-inference"
INFERENCE_ENVIRONMENT_VERSION = "2"
INSTANCE_TYPE = "Standard_NC4as_T4_v3"
MIN_ANSWER_TOKEN_F1 = 0.60
MAX_MEAN_LATENCY_MS = 20_000
RUN_SHAP = False
ARTIFACT_DIR = PROJECT_ROOT / "azureml-artifacts"

assert "REPLACE" not in REGISTERED_MODEL_VERSION
assert "REPLACE" not in ENDPOINT_NAME

## Connect and deploy the candidate

The managed endpoint uses short-lived Azure ML token authentication. Request and response collection is enabled for observability; enforce private networking, diagnostic settings, RBAC, quotas, and a deletion policy in the target environment.

In [3]:
config = AzureMLConfig.from_env()
ml_client = config.create_ml_client()
registered_model = ml_client.models.get(
    REGISTERED_MODEL_NAME, version=REGISTERED_MODEL_VERSION
)
inference_environment = create_inference_environment(
    ml_client, INFERENCE_ENVIRONMENT_NAME, INFERENCE_ENVIRONMENT_VERSION
)
endpoint = deploy_managed_endpoint(
    ml_client=ml_client,
    endpoint_name=ENDPOINT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    model=registered_model.id,
    environment=inference_environment.id,
    instance_type=INSTANCE_TYPE,
    traffic_percent=0,
)
print("Candidate deployed with zero traffic:", endpoint.scoring_uri)

ResourceNotFoundError: (UserError) The specified resource was not found.
Code: UserError
Message: The specified resource was not found.
Exception Details:	(NoSuchModelRegistered) There is no registered model in Account Subscription: ff9fa810-9dbb-4085-9c75-10b2f491bace, ResourceGroup: sriks-mlhub-mcaps, Workspace: sriks-aml-ws with id raft-llama32-1b:raft-llama32-1b:1
	Code: NoSuchModelRegistered
	Message: There is no registered model in Account Subscription: ff9fa810-9dbb-4085-9c75-10b2f491bace, ResourceGroup: sriks-mlhub-mcaps, Workspace: sriks-aml-ws with id raft-llama32-1b:raft-llama32-1b:1

## Readiness smoke test

Invoke the named deployment directly. This works before traffic promotion and isolates candidate behavior from the current production deployment.

In [ ]:
test_records = read_jsonl(PROJECT_ROOT / "data" / "training_data_raft" / "test.jsonl")
smoke_result = invoke_endpoint(
    ml_client,
    ENDPOINT_NAME,
    [{"instruction": test_records[0]["instruction"], "max_new_tokens": 128}],
    deployment_name=DEPLOYMENT_NAME,
)
smoke_result

## Held-out evaluation and MLflow evidence

Token-F1 and exact match operate on the `<ANSWER>` span, not the explanatory reasoning. Latency measured inside scoring excludes network time; also run concurrent end-to-end load tests and inspect p50/p95/p99 before setting service objectives.

In [ ]:
evaluated_rows, evaluation_metrics = evaluate_endpoint(
    ml_client,
    ENDPOINT_NAME,
    test_records,
    batch_size=1,
    deployment_name=DEPLOYMENT_NAME,
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
evaluation_path = ARTIFACT_DIR / "held_out_evaluation.jsonl"
evaluation_path.write_text(
    "".join(json.dumps(row) + "\n" for row in evaluated_rows),
    encoding="utf-8",
)

workspace = ml_client.workspaces.get(config.workspace_name)
mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
mlflow.set_experiment("raft-production-evaluation")
with mlflow.start_run(run_name=f"{DEPLOYMENT_NAME}-held-out"):
    mlflow.log_metrics(evaluation_metrics)
    mlflow.log_params({
        "model_name": REGISTERED_MODEL_NAME,
        "model_version": REGISTERED_MODEL_VERSION,
        "endpoint": ENDPOINT_NAME,
        "deployment": DEPLOYMENT_NAME,
        "test_rows": len(test_records),
    })
    mlflow.log_artifact(str(evaluation_path), artifact_path="evaluation")
evaluation_metrics

## Explicit promotion gate

Promotion is intentionally separate from deployment. A failed gate leaves the candidate at zero traffic for investigation or deletion. For an existing endpoint, use staged traffic and automated rollback rather than replacing all traffic at once.

In [ ]:
quality_passed = evaluation_metrics["answer_token_f1"] >= MIN_ANSWER_TOKEN_F1
latency_passed = evaluation_metrics.get("mean_latency_ms", float("inf")) <= MAX_MEAN_LATENCY_MS
if not (quality_passed and latency_passed):
    raise RuntimeError({
        "message": "Candidate did not satisfy the promotion gate",
        "metrics": evaluation_metrics,
    })

promoted_endpoint = promote_deployment(ml_client, ENDPOINT_NAME, DEPLOYMENT_NAME)
print("Production traffic:", promoted_endpoint.traffic)

## SHAP diagnostic for answer quality

The score function sends masked prompts to the candidate and compares each generated answer with one fixed reference. Run this on a small, approved diagnostic sample because it requires many endpoint calls. Store the target definition, model version, sample ID, masker, and SHAP version with the artifact.

In [ ]:
if RUN_SHAP:
    from transformers import AutoTokenizer

    sample = test_records[0]
    shap_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

    def endpoint_quality_score(masked_instructions):
        predictions = invoke_endpoint(
            ml_client,
            ENDPOINT_NAME,
            [{"instruction": value, "max_new_tokens": 256} for value in masked_instructions],
            deployment_name=DEPLOYMENT_NAME,
        )
        return [
            token_f1(result["prediction"], sample["cot_answer"])
            for result in predictions
        ]

    shap_values = explain_text_score(
        [sample["instruction"]],
        endpoint_quality_score,
        shap_tokenizer,
        max_evals=100,
    )
    display(shap_values)
else:
    print("Set RUN_SHAP=True after endpoint cost approval.")

## Drift baseline and detection

The reference profile describes prompt length, question length, and retrieved-document count. Population Stability Index (PSI) is a triage signal: investigate at `0.1`, alert at `0.2`, and retrain only after confirming a sustained, consequential shift. Monitor outcome quality, abstention, safety, latency, errors, and cost alongside feature drift.

In [ ]:
reference_profile = create_reference_profile(test_records)
reference_path = write_json_artifact(
    reference_profile, ARTIFACT_DIR / "reference_profile.json"
)

# Replace this sample with privacy-reviewed request logs exported from endpoint collection.
production_sample = test_records
drift_report = detect_drift(reference_profile, production_sample, psi_threshold=0.2)
drift_path = write_json_artifact(
    drift_report, ARTIFACT_DIR / "drift_report.json"
)
pd.DataFrame(drift_report["features"]).T

## Schedule the drift job

Set `PRODUCTION_DATA_URI` to a governed Azure Storage path containing JSONL request features collected for the monitoring window. The scheduled command logs PSI metrics and a JSON report to MLflow. Apply retention and access policies because prompts may contain sensitive data.

In [ ]:
from azure.ai.ml import Input, Output, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import JobSchedule, RecurrencePattern, RecurrenceTrigger

CREATE_DRIFT_SCHEDULE = False
PRODUCTION_DATA_URI = "REPLACE_WITH_GOVERNED_AZUREML_OR_STORAGE_URI"
MONITOR_COMPUTE = "cpu-cluster"

if CREATE_DRIFT_SCHEDULE:
    monitor_job = command(
        code=str(PROJECT_ROOT),
        command=(
            "python -m lib.monitor_drift "
            "--reference-profile ${{inputs.reference}} "
            "--production-data ${{inputs.production}} "
            "--output ${{outputs.report}} --psi-threshold 0.2"
        ),
        inputs={
            "reference": Input(type=AssetTypes.URI_FILE, path=str(reference_path)),
            "production": Input(type=AssetTypes.URI_FOLDER, path=PRODUCTION_DATA_URI),
        },
        outputs={"report": Output(type=AssetTypes.URI_FOLDER)},
        environment=inference_environment.id,
        compute=MONITOR_COMPUTE,
        experiment_name="raft-production-drift",
    )
    trigger = RecurrenceTrigger(
        frequency="week",
        interval=1,
        schedule=RecurrencePattern(hours=2, minutes=0),
        time_zone="UTC",
    )
    schedule = JobSchedule(
        name=f"{ENDPOINT_NAME}-weekly-drift", trigger=trigger, create_job=monitor_job
    )
    ml_client.schedules.begin_create_or_update(schedule).result()
else:
    print("Review the production log URI and privacy controls before scheduling.")

## Operational checklist

Before production approval, verify private endpoint/network policy, managed identities and least privilege, encryption and secret rotation, prompt redaction, abuse controls, quotas, autoscaling, availability-zone strategy, p95/p99 load tests, Application Insights alerts, model/data cards, incident ownership, rollback rehearsal, retention, license terms, and budget alerts. Monitor quality by RAFT sample type so aggregate metrics do not hide abstention regressions.